# 5. XGBoost Hyperparameter Tuning

This notebook improves the XGBoost model developed during model comparison.

The main objectives are:

- Load the prepared training and testing datasets
- Handle class imbalance using `scale_pos_weight`
- Build an XGBoost preprocessing and modelling pipeline
- Perform hyperparameter tuning using RandomizedSearchCV
- Use cross-validation on the training data
- Evaluate the tuned model on the untouched test set
- Compare the tuned model with the original XGBoost baseline

The test dataset is not used during hyperparameter selection.

In [1]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer

from sklearn.model_selection import (
    RandomizedSearchCV,
    StratifiedKFold
)

from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    brier_score_loss,
    log_loss,
    confusion_matrix,
    classification_report
)

from xgboost import XGBClassifier

import joblib

pd.set_option("display.max_columns", None)

#### Load processed data

In [2]:
X_train = pd.read_csv("../data/processed/X_train.csv")
X_test = pd.read_csv("../data/processed/X_test.csv")

y_train = pd.read_csv("../data/processed/y_train.csv").squeeze()
y_test = pd.read_csv("../data/processed/y_test.csv").squeeze()

print("Data loaded successfully.")

print("\nX_train:", X_train.shape)
print("X_test:", X_test.shape)
print("y_train:", y_train.shape)
print("y_test:", y_test.shape)

Data loaded successfully.

X_train: (24000, 39)
X_test: (6000, 39)
y_train: (24000,)
y_test: (6000,)


#### Check target distribution

In [3]:
print("Training target distribution:")
print(y_train.value_counts())

print("\nTraining target proportions:")
print(y_train.value_counts(normalize=True))

Training target distribution:
default payment next month
0    18691
1     5309
Name: count, dtype: int64

Training target proportions:
default payment next month
0    0.778792
1    0.221208
Name: proportion, dtype: float64


#### Calculate class imbalance ratio

In [4]:
negative_count = (y_train == 0).sum()
positive_count = (y_train == 1).sum()

scale_pos_weight = negative_count / positive_count

print("Non-default cases:", negative_count)
print("Default cases:", positive_count)
print("Scale positive weight:", scale_pos_weight)

Non-default cases: 18691
Default cases: 5309
Scale positive weight: 3.520625353173856


#### Define feature groups

In [5]:
categorical_features = [
    "SEX",
    "EDUCATION",
    "MARRIAGE"
]

numerical_features = [
    col for col in X_train.columns
    if col not in categorical_features
]

print("Categorical features:")
print(categorical_features)

print("\nNumber of numerical features:", len(numerical_features))
print("Number of categorical features:", len(categorical_features))
print("Total features:", len(numerical_features) + len(categorical_features))

Categorical features:
['SEX', 'EDUCATION', 'MARRIAGE']

Number of numerical features: 36
Number of categorical features: 3
Total features: 39


#### Create preprocessing pipeline

In [6]:
numeric_pipeline = Pipeline([
    (
        "imputer",
        SimpleImputer(strategy="median")
    )
])

categorical_pipeline = Pipeline([
    (
        "imputer",
        SimpleImputer(strategy="most_frequent")
    ),
    (
        "onehot",
        OneHotEncoder(handle_unknown="ignore")
    )
])

In [7]:
# Create ColumnTransformer

preprocessor = ColumnTransformer([
    (
        "num",
        numeric_pipeline,
        numerical_features
    ),
    (
        "cat",
        categorical_pipeline,
        categorical_features
    )
])

#### Create XGBoost model

In [8]:
xgb_classifier = XGBClassifier(
    objective="binary:logistic",
    eval_metric="logloss",
    random_state=42,
    n_jobs=-1,

    scale_pos_weight=scale_pos_weight
)

#### Create complete pipeline

In [9]:
xgb_pipeline = Pipeline([
    (
        "preprocessor",
        preprocessor
    ),
    (
        "classifier",
        xgb_classifier
    )
])

print("XGBoost pipeline created successfully.")

XGBoost pipeline created successfully.


#### Define hyperparameter search space

In [10]:
param_distributions = {
    "classifier__n_estimators": [
        200,
        300,
        500,
        700
    ],

    "classifier__max_depth": [
        3,
        4,
        5,
        6,
        7
    ],

    "classifier__learning_rate": [
        0.02,
        0.05,
        0.08,
        0.10
    ],

    "classifier__min_child_weight": [
        1,
        3,
        5,
        7
    ],

    "classifier__subsample": [
        0.7,
        0.8,
        0.9,
        1.0
    ],

    "classifier__colsample_bytree": [
        0.7,
        0.8,
        0.9,
        1.0
    ],

    "classifier__gamma": [
        0,
        0.1,
        0.3,
        0.5
    ],

    "classifier__reg_alpha": [
        0,
        0.01,
        0.1,
        1
    ],

    "classifier__reg_lambda": [
        1,
        2,
        5,
        10
    ]
}

#### Create cross-validation strategy

In [11]:
cv_strategy = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

#### Create RandomizedSearchCV

In [12]:
random_search = RandomizedSearchCV(
    estimator=xgb_pipeline,
    param_distributions=param_distributions,

    n_iter=30,

    scoring="average_precision",

    cv=cv_strategy,

    verbose=2,

    random_state=42,

    n_jobs=-1,

    return_train_score=True
)

####  Tunning

In [13]:
print("Starting XGBoost hyperparameter tuning...")
print("This may take some time.")

random_search.fit(
    X_train,
    y_train
)

print("\nHyperparameter tuning completed.")

Starting XGBoost hyperparameter tuning...
This may take some time.
Fitting 5 folds for each of 30 candidates, totalling 150 fits

Hyperparameter tuning completed.


#### Best parameters

In [14]:
print("Best parameters:")
print(random_search.best_params_)

Best parameters:
{'classifier__subsample': 0.7, 'classifier__reg_lambda': 2, 'classifier__reg_alpha': 0.01, 'classifier__n_estimators': 200, 'classifier__min_child_weight': 1, 'classifier__max_depth': 6, 'classifier__learning_rate': 0.02, 'classifier__gamma': 0.3, 'classifier__colsample_bytree': 0.9}


#### Best cross-validation score

In [15]:
print("Best cross-validation PR-AUC:")
print(random_search.best_score_)

Best cross-validation PR-AUC:
0.5661974033987356


#### Create results table

In [16]:
cv_results = pd.DataFrame(
    random_search.cv_results_
)

cv_results = cv_results.sort_values(
    by="mean_test_score",
    ascending=False
)

cv_results[
    [
        "rank_test_score",
        "mean_test_score",
        "std_test_score",
        "mean_train_score",
        "params"
    ]
].head(10)

,rank_test_score,mean_test_score,std_test_score,mean_train_score,params
6,1,0.566197,0.007275,0.679540,"{'classifier__subsample': 0.7, 'classifier__re..."
14,2,0.565609,0.007687,0.666495,"{'classifier__subsample': 0.7, 'classifier__re..."
11,3,0.565182,0.006217,0.654911,"{'classifier__subsample': 1.0, 'classifier__re..."
21,4,0.565171,0.006592,0.626303,"{'classifier__subsample': 1.0, 'classifier__re..."
19,5,0.564795,0.007352,0.635189,"{'classifier__subsample': 0.8, 'classifier__re..."
13,6,0.562890,0.010163,0.602235,"{'classifier__subsample': 0.9, 'classifier__re..."
1,7,0.561383,0.005035,0.748191,"{'classifier__subsample': 1.0, 'classifier__re..."
10,8,0.561287,0.003673,0.703258,"{'classifier__subsample': 1.0, 'classifier__re..."
22,9,0.561109,0.011492,0.596665,"{'classifier__subsample': 1.0, 'classifier__re..."
25,10,0.560655,0.010213,0.606365,"{'classifier__subsample': 1.0, 'classifier__re..."


#### Get best model

In [17]:
best_xgb_model = random_search.best_estimator_

print("Best XGBoost model selected.")

Best XGBoost model selected.


#### Predict on test data

In [18]:
tuned_pred = best_xgb_model.predict(X_test)

tuned_proba = best_xgb_model.predict_proba(X_test)[:, 1]

#### Calculate performance metrics

In [19]:
tuned_accuracy = accuracy_score(
    y_test,
    tuned_pred
)

tuned_balanced_accuracy = balanced_accuracy_score(
    y_test,
    tuned_pred
)

tuned_precision = precision_score(
    y_test,
    tuned_pred
)

tuned_recall = recall_score(
    y_test,
    tuned_pred
)

tuned_f1 = f1_score(
    y_test,
    tuned_pred
)

tuned_roc_auc = roc_auc_score(
    y_test,
    tuned_proba
)

tuned_pr_auc = average_precision_score(
    y_test,
    tuned_proba
)

tuned_brier = brier_score_loss(
    y_test,
    tuned_proba
)

tuned_log_loss = log_loss(
    y_test,
    tuned_proba
)

#### Display performance

In [20]:
print("Tuned XGBoost Performance")
print("=" * 45)

print(f"Accuracy:           {tuned_accuracy:.4f}")
print(f"Balanced Accuracy:  {tuned_balanced_accuracy:.4f}")
print(f"Precision:          {tuned_precision:.4f}")
print(f"Recall:             {tuned_recall:.4f}")
print(f"F1 Score:           {tuned_f1:.4f}")
print(f"ROC-AUC:            {tuned_roc_auc:.4f}")
print(f"PR-AUC:             {tuned_pr_auc:.4f}")
print(f"Brier Score:        {tuned_brier:.4f}")
print(f"Log Loss:           {tuned_log_loss:.4f}")

Tuned XGBoost Performance
Accuracy:           0.7630
Balanced Accuracy:  0.7116
Precision:          0.4727
Recall:             0.6194
F1 Score:           0.5362
ROC-AUC:            0.7823
PR-AUC:             0.5622
Brier Score:        0.1761
Log Loss:           0.5346


#### Confusion matrix

In [21]:
cm = confusion_matrix(
    y_test,
    tuned_pred
)

print("Confusion Matrix:")
print(cm)

Confusion Matrix:
[[3756  917]
 [ 505  822]]


In [22]:
print("\nClassification Report:")
print(
    classification_report(
        y_test,
        tuned_pred,
        target_names=[
            "Non-Default",
            "Default"
        ]
    )
)


Classification Report:
              precision    recall  f1-score   support

 Non-Default       0.88      0.80      0.84      4673
     Default       0.47      0.62      0.54      1327

    accuracy                           0.76      6000
   macro avg       0.68      0.71      0.69      6000
weighted avg       0.79      0.76      0.77      6000



#### Probability statistics

In [23]:
print("Predicted probability statistics:")
print(
    pd.Series(tuned_proba).describe()
)

print("\nFirst 10 predicted default probabilities:")
print(tuned_proba[:10])


Predicted probability statistics:
count    6000.000000
mean        0.415398
std         0.225757
min         0.069196
25%         0.242656
50%         0.364205
75%         0.545162
max         0.946515
dtype: float64

First 10 predicted default probabilities:
[0.32368588 0.35758886 0.35819525 0.36671624 0.11318757 0.6299751
 0.27917844 0.09174963 0.27840567 0.18243392]


#### Compare with original XGBoost

In [24]:
original_xgb = {
    "Model": "Original XGBoost",
    "Accuracy": 0.816167,
    "Balanced Accuracy": 0.654547,
    "Precision": 0.650538,
    "Recall": 0.364732,
    "F1": 0.467407,
    "ROC-AUC": 0.779569,
    "PR-AUC": 0.554921,
    "Brier Score": 0.135553,
    "Log Loss": 0.431365
}

In [25]:
tuned_xgb = {
    "Model": "Tuned XGBoost",
    "Accuracy": tuned_accuracy,
    "Balanced Accuracy": tuned_balanced_accuracy,
    "Precision": tuned_precision,
    "Recall": tuned_recall,
    "F1": tuned_f1,
    "ROC-AUC": tuned_roc_auc,
    "PR-AUC": tuned_pr_auc,
    "Brier Score": tuned_brier,
    "Log Loss": tuned_log_loss
}

comparison = pd.DataFrame([
    original_xgb,
    tuned_xgb
])

comparison

,Model,Accuracy,Balanced Accuracy,Precision,Recall,F1,ROC-AUC,PR-AUC,Brier Score,Log Loss
0,Original XGBoost,0.816167,0.654547,0.650538,0.364732,0.467407,0.779569,0.554921,0.135553,0.431365
1,Tuned XGBoost,0.763000,0.711604,0.472685,0.619442,0.536204,0.782300,0.562165,0.176100,0.534610


#### Calculate improvement

In [26]:
improvement = pd.DataFrame({
    "Metric": [
        "Accuracy",
        "Balanced Accuracy",
        "Precision",
        "Recall",
        "F1",
        "ROC-AUC",
        "PR-AUC",
        "Brier Score",
        "Log Loss"
    ],

    "Original XGBoost": [
        0.816167,
        0.654547,
        0.650538,
        0.364732,
        0.467407,
        0.779569,
        0.554921,
        0.135553,
        0.431365
    ],

    "Tuned XGBoost": [
        tuned_accuracy,
        tuned_balanced_accuracy,
        tuned_precision,
        tuned_recall,
        tuned_f1,
        tuned_roc_auc,
        tuned_pr_auc,
        tuned_brier,
        tuned_log_loss
    ]
})

improvement["Difference"] = (
    improvement["Tuned XGBoost"]
    - improvement["Original XGBoost"]
)

improvement

,Metric,Original XGBoost,Tuned XGBoost,Difference
0,Accuracy,0.816167,0.763000,-0.053167
1,Balanced Accuracy,0.654547,0.711604,0.057057
2,Precision,0.650538,0.472685,-0.177853
3,Recall,0.364732,0.619442,0.254710
4,F1,0.467407,0.536204,0.068797
5,ROC-AUC,0.779569,0.782300,0.002731
6,PR-AUC,0.554921,0.562165,0.007244
7,Brier Score,0.135553,0.176100,0.040547
8,Log Loss,0.431365,0.534610,0.103245


#### Save tuning results

In [27]:
import os

os.makedirs("../reports", exist_ok=True)

comparison.to_csv(
    "../reports/xgboost_model_comparison.csv",
    index=False
)

cv_results.to_csv(
    "../reports/xgboost_tuning_results.csv",
    index=False
)

print("Results saved successfully.")

Results saved successfully.


In [28]:
# Save the tunned model for future use

import os
import joblib

os.makedirs("../models", exist_ok=True)

joblib.dump(random_search.best_estimator_, "../models/tuned_xgboost_model.pkl")

print("Tuned XGBoost model saved successfully.")

Tuned XGBoost model saved successfully.
